# SD1.5 K-Tilde Lambda Comparison

This notebook audits the SD1.5 k-tilde prior bank before those priors are used in the recovery experiments. It recreates the Algorithm 1 convergence exercise, loads the configured k-tilde artifacts, summarizes which priors exist, computes the empirical baseline complexity `kappa` for each recovery class, and computes the cross-prior `lambda` and mismatch-penalty tables that measure how costly it is to sample with one prior while reconstructing another class.

Run it top to bottom when you want to check that the prior bank is complete and export publication-ready compatibility figures. The next cell resolves the project root, loads `sd15_conditioning_experiment.py`, builds the numeric tables, displays the unitary-normalized values, and prints an interpretation of the row-normalized mismatch factor.

Absolute `lambda` and `kappa` metrics are reported in the unitary-FFT convention by dividing the stored unnormalized-FFT `K_tilde` energies by `H * W`.

In [ ]:
from pathlib import Path
import sys

import importlib
import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_DIR = Path.cwd()
for candidate in [NOTEBOOK_DIR, *NOTEBOOK_DIR.parents]:
    helper_candidates = [
        candidate / 'sd15_conditioning_experiment.py',
        candidate / 'sd1.5' / 'analyze_results' / 'sd15_conditioning_experiment.py',
    ]
    for helper_path in helper_candidates:
        if helper_path.is_file():
            helper_dir = helper_path.parent
            if str(helper_dir) not in sys.path:
                sys.path.insert(0, str(helper_dir))
            break
    else:
        continue
    break
else:
    raise FileNotFoundError('Could not find sd15_conditioning_experiment.py from the notebook cwd.')

import sd15_conditioning_experiment as exp
exp = importlib.reload(exp)

SD15_ROOT = exp.find_sd15_root(NOTEBOOK_DIR)
CATALOG = exp.load_ktilde_catalog(SD15_ROOT)
TABLES = exp.build_lambda_tables(SD15_ROOT, skip_missing=True)
BANK_SUMMARY = pd.DataFrame(
    [
        {
            'name': name,
            'role': value.get('role', name),
            'label': value.get('label', name),
            'christoffel_law': value.get('christoffel_law', ''),
            'prompt': value.get('prompt', ''),
            'prompt_bank': ', '.join(value.get('prompt_bank', [])),
            'artifact_exists': (SD15_ROOT / 'ktilde' / 'unweighted' / f'{name}.npz').is_file(),
        }
        for name, value in CATALOG.items()
    ]
)


def style_plain_numbers(frame):
    return frame.style.format(lambda value: exp.format_plain_number(value))


display(BANK_SUMMARY)
if TABLES['missing_names']:
    print('Missing k-tilde artifacts:', TABLES['missing_names'])
FFT_ENERGY_SCALES = sorted({int(round(value)) for value in TABLES['fft_energy_scale'].values()})
display(
    Markdown(
        "**FFT normalization.** "
        "Stored `K_tilde` artifacts were estimated with the unnormalized FFT, so the absolute lambda and kappa tables below divide Fourier energies by `H * W` "
        f"({', '.join(str(value) for value in FFT_ENERGY_SCALES)}) to report the unitary-FFT convention. "
        "The mismatch table is unchanged because this scale cancels."
    )
)
display(TABLES['kappa_df'].style.format({'kappa_hat': exp.format_plain_number}))
display(
    TABLES['lambda_df'].style.format(
        {
            'lambda_hat': exp.format_plain_number,
            'kappa_hat': exp.format_plain_number,
            'mismatch_penalty': exp.format_plain_number,
        }
    )
)
display(style_plain_numbers(TABLES['lambda_table']))
display(style_plain_numbers(TABLES['penalty_table']))
display(
    TABLES['matched_check'].style.format(
        {
            'lambda_hat': exp.format_plain_number,
            'kappa_hat': exp.format_plain_number,
            'abs_lambda_minus_kappa': exp.format_plain_number,
        }
    )
)
display(
    Markdown(
        "**$\widetilde{\Lambda}'$ interpretation.** "
        "The $\widetilde{\lambda}$ heatmap shows the absolute compatibility cost of using sampling law "
        "$\widetilde{\mu}_{c_s}$ with Christoffel function $c_r$. "
        "$\widetilde{\Lambda}'$ is the row-normalized mismatch factor "
        "$\widetilde{\lambda}(c_r,c_r,c_s)/\widetilde{\kappa}(c_r)$, so values near 1 mean the sampling law is close to the matched baseline for that row, while larger values show how much extra penalty you pay from mismatch."
    )
)


## Export Lambda Figures

This cell exports the absolute Lambda heatmap, the individual sampling-law plots, and the compact sampling-law row under `results/unweighted/ktilde/figures`.

The displayed `FIGURE_PATHS` series is the checklist of files produced by the helper, so it is the quickest way to confirm where the notebook saved each figure.

In [ ]:
FIGURE_DIR = SD15_ROOT / 'results' / 'unweighted' / 'ktilde' / 'figures'
FIGURE_PATHS = exp.export_lambda_figure_set(
    TABLES,
    output_dir=FIGURE_DIR,
    file_format='pdf',
    show=True,
)
display(pd.Series({key: str(value) for key, value in FIGURE_PATHS.items()}))
FIGURE_PATHS


## CFG Sampling-Distribution Grid

This cell visualizes the actual k-tilde sampling probability maps used by the CFG-scale ablation. Rows are the conditioned prior families, columns are CFG 1, CFG 3, CFG 5, and CFG 7.5, and every panel shares one logarithmic color scale so the distributions can be compared visually.

The cell loads `ktilde/unweighted/config_cfg_ablation.json` plus the matching `.npz` artifacts. If the lower-CFG artifacts have not been generated yet, it raises an error with the build-script names to run; otherwise it saves the grid PDF into the same lambda-figure output directory.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LogNorm
from matplotlib.ticker import LogFormatterMathtext

def load_ktilde_npz(path):
    data = np.load(str(path), allow_pickle=False)
    k_tilde = data['K_tilde'].astype(np.float64)
    probabilities = data['prob'].astype(np.float64)
    metadata = json.loads(str(data['meta']))
    return k_tilde, probabilities, metadata

CFG_DISTRIBUTION_CONFIG = SD15_ROOT / 'ktilde' / 'unweighted' / 'config_cfg_ablation.json'
with CFG_DISTRIBUTION_CONFIG.open('r', encoding='utf-8') as handle:
    CFG_DISTRIBUTION_CATALOG = json.load(handle)['ktilde']
FIGURE_DIR = SD15_ROOT / 'results' / 'unweighted' / 'ktilde' / 'figures'

CFG_COLUMNS = [
    ('cfg1', 'CFG 1'),
    ('cfg3', 'CFG 3'),
    ('cfg5', 'CFG 5'),
    ('cfg7p5', 'CFG 7.5'),
]
CFG_ROWS = [
    ('k1daytimebeach', r'$\widetilde{\mu}_{c_{\mathrm{db}}}$'),
    ('k2sunsetbeach', r'$\widetilde{\mu}_{c_{\mathrm{sb}}}$'),
    ('k4cat', r'$\widetilde{\mu}_{c_{\mathrm{ca}}}$'),
]


def cfg_distribution_name(row_key, cfg_key):
    if cfg_key == 'cfg7p5':
        return f'Ktilde_SD15__fft__{row_key}_512x512_S500_ns20'
    return f'Ktilde_SD15__fft__{row_key}_{cfg_key}_512x512_S500_ns20'


def reshape_distribution(probabilities, metadata):
    height = int(metadata.get('height', int(np.sqrt(probabilities.size))))
    width = int(metadata.get('width', int(probabilities.size // max(height, 1))))
    return np.asarray(probabilities, dtype=np.float64).reshape(height, width)


grid_distributions = {}
missing_names = []
for row_key, _ in CFG_ROWS:
    for cfg_key, _ in CFG_COLUMNS:
        name = cfg_distribution_name(row_key, cfg_key)
        if cfg_key != 'cfg7p5' and name not in CFG_DISTRIBUTION_CATALOG:
            raise KeyError(f'Missing {name} from {CFG_DISTRIBUTION_CONFIG}')
        artifact_path = SD15_ROOT / 'ktilde' / 'unweighted' / f'{name}.npz'
        if not artifact_path.is_file():
            missing_names.append(name)
            continue
        _, probabilities, metadata = load_ktilde_npz(artifact_path)
        grid_distributions[(row_key, cfg_key)] = reshape_distribution(probabilities, metadata)

if missing_names:
    script_hint = (
        'Run scripts/unweighted/ktilde/cfg_ablation/build_k1_daytime_beach_cfg1.sh and the '
        'matching cfg1/cfg3/cfg5 scripts under scripts/unweighted/ktilde/cfg_ablation first. '
        'The CFG 7.5 column uses the default prebuilt k-tilde artifacts.'
    )
    missing_block = '\n'.join(f'  - {name}' for name in missing_names)
    raise FileNotFoundError(f'Missing CFG k-tilde artifacts. {script_hint}\n{missing_block}')

positive_values = np.concatenate([image[image > 0.0].reshape(-1) for image in grid_distributions.values()])
norm = LogNorm(vmin=float(positive_values.min()), vmax=float(positive_values.max()))
cmap = LinearSegmentedColormap.from_list('cfg_distribution_grid', exp.SD15_DISTRIBUTION_CMAP_COLORS)
CFG_GRID_PATH = FIGURE_DIR / 'ktilde_sampling_distribution_cfg1_cfg3_cfg5_cfg7p5_grid.pdf'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

with plt.rc_context(exp.SD15_PRESENTATION_RC):
    fig, axes = plt.subplots(
        len(CFG_ROWS),
        len(CFG_COLUMNS),
        figsize=(12.2, 9.2),
        constrained_layout=True,
    )
    axes = np.asarray(axes, dtype=object)
    fig.set_constrained_layout_pads(w_pad=0.03, h_pad=0.03, wspace=0.02, hspace=0.02)
    image_artist = None
    for row_idx, (row_key, row_label) in enumerate(CFG_ROWS):
        for col_idx, (cfg_key, cfg_label) in enumerate(CFG_COLUMNS):
            ax = axes[row_idx, col_idx]
            image_artist = ax.imshow(
                grid_distributions[(row_key, cfg_key)],
                cmap=cmap,
                norm=norm,
                interpolation='nearest',
            )
            ax.set_xticks([])
            ax.set_yticks([])
            if row_idx == 0:
                ax.set_title(cfg_label, pad=12)
            if col_idx == 0:
                ax.set_ylabel(row_label, rotation=0, ha='right', va='center', labelpad=15)
            for spine in ax.spines.values():
                spine.set_visible(False)

    colorbar = fig.colorbar(image_artist, ax=axes.reshape(-1).tolist(), fraction=0.035, pad=0.02, aspect=32)
    colorbar.ax.yaxis.set_major_formatter(LogFormatterMathtext())
    colorbar.ax.tick_params(labelsize=13, width=0.9, length=4)
    colorbar.outline.set_linewidth(0.8)
    fig.savefig(CFG_GRID_PATH, dpi=exp.SD15_EXPORT_DPI, bbox_inches='tight')
    display(fig)
    plt.close(fig)

display(pd.Series({'cfg_distribution_grid': str(CFG_GRID_PATH)}))
CFG_GRID_PATH
